# Projection Accuracy Analysis


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, HTML

pd.set_option('display.float_format', '{:.2%}'.format)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid', palette='muted')

In [2]:
FILE_PATH = '../data/processed/projection_dblp_50.csv'   # <-- change this to your file path

df = pd.read_csv(FILE_PATH)

# Normalise the match column: accept both boolean and string variants
if df['match'].dtype == object:
    df['match'] = df['match'].str.strip().str.lower().map({'true': True, 'false': False})
else:
    df['match'] = df['match'].astype(bool)

print(f"Rows: {len(df):,}  |  Columns: {list(df.columns)}")
df.head()

Rows: 5,500  |  Columns: ['m', 'col', 'model', 'row_id', 'match', 'true_value', 'projected_value', 'repetition']


,m,col,model,row_id,match,true_value,projected_value,repetition
0,1,authors,EmbDI_300,798,True,l_forlizzi_r_gting_e_nardelli_m_schneider,l_forlizzi_r_gting_e_nardelli_m_schneider,1
1,1,authors,EmbDI_300,1872,True,l_gravano_h_garciamolina,l_gravano_h_garciamolina,1
2,1,authors,EmbDI_300,2497,True,m_mecella_b_pernici,m_mecella_b_pernici,1
3,1,authors,EmbDI_300,3561,True,rs_jasinschi,rs_jasinschi,1
4,1,authors,EmbDI_300,4421,True,d_tannen,d_tannen,1


In [3]:
print("=== Dataset Overview ===")
print(f"  Total predictions : {len(df):,}")
print(f"  Unique models     : {df['model'].nunique()} → {sorted(df['model'].unique())}")
print(f"  Unique columns    : {df['col'].nunique()} → {sorted(df['col'].unique())}")
print(f"  Repetitions       : {sorted(df['repetition'].unique())}")
print(f"  Overall accuracy  : {df['match'].mean():.2%}")
print()
print("Missing values:")
display(df.isnull().sum().rename('nulls').to_frame())

=== Dataset Overview ===
  Total predictions : 5,500
  Unique models     : 5 → ['EmbDI_300', 'EmbDI_512', 'HRR_1024', 'HRR_300', 'HRR_512']
  Unique columns    : 4 → ['authors', 'title', 'venue', 'year']
  Repetitions       : [0, 1, 2]
  Overall accuracy  : 98.60%

Missing values:


,nulls
m,0
col,0
model,0
row_id,0
match,0
true_value,627
projected_value,558
repetition,0


## Accuracy per Model x Column


In [4]:
def plot_model_col(dfr, m_value):
    d_frame = dfr.copy()

    model_list = [
    # "BSC_300",
    # "BSC_512",
    # "BSC_1024",
    # "BSC_2048",
    # "BSC_4096",
    # "BSC_8192",
    "EmbDI_300",
    "EmbDI_512",
    "HRR_300",
    "HRR_512",
    # "HRR_1024",
    # "HRR_2048",
    # "SemHDC_FastText_300"
    ]
    
    df_m = d_frame[(d_frame['m']== m_value) & (d_frame["model"].isin(model_list))]

    # Pivot: rows = model, columns = col
    acc_pivot = (
        df_m.groupby(['model', 'col'])['match']
        .agg(accuracy='mean', n_predictions='count', n_correct='sum')
        .reset_index()
    )

    pivot_table = acc_pivot.pivot(index='model', columns='col', values='accuracy')


    # Reorder rows to match model_list, keeping any unexpected models at the end
    ordered_models = [model for model in model_list if model in pivot_table.index]
    remaining_models = [model for model in pivot_table.index if model not in ordered_models]
    pivot_table = pivot_table.reindex(ordered_models + remaining_models)

    # Add row-level aggregates
    pivot_table.insert(0, 'OVERALL', df_m.groupby('model')['match'].mean())
    # pivot_table['BEST_COL']  = pivot_table.drop(columns='OVERALL').idxmax(axis=1)
    # pivot_table['WORST_COL'] = pivot_table.drop(columns=['OVERALL', 'BEST_COL']).idxmin(axis=1)

    # Colour the numeric cells
    numeric_cols = [c for c in pivot_table.columns if c not in ('BEST_COL', 'WORST_COL')]

    styled = (
        pivot_table.style
        .format({c: '{:.2%}' for c in numeric_cols})
        .background_gradient(cmap='RdYlGn', subset=numeric_cols, vmin=0, vmax=1)
        .set_caption(f'[m={m_value}] Accuracy by Model x Column')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold'),
                                    ('text-align', 'left'), ('padding-bottom', '6px')]}])
    )
    display(styled)
    print(pivot_table.round(2).to_latex(float_format="%.2f"))

In [5]:
for m in sorted(df['m'].unique()):
    plot_model_col(df, m)

col,OVERALL,authors
model,,
EmbDI_300,100.00%,100.00%
EmbDI_512,100.00%,100.00%
HRR_300,100.00%,100.00%
HRR_512,100.00%,100.00%


\begin{tabular}{lrr}
\toprule
col & OVERALL & authors \\
model &  &  \\
\midrule
EmbDI_300 & 1.00 & 1.00 \\
EmbDI_512 & 1.00 & 1.00 \\
HRR_300 & 1.00 & 1.00 \\
HRR_512 & 1.00 & 1.00 \\
\bottomrule
\end{tabular}



col,OVERALL,authors,title
model,,,
EmbDI_300,100.00%,100.00%,100.00%
EmbDI_512,100.00%,100.00%,100.00%
HRR_300,100.00%,100.00%,100.00%
HRR_512,100.00%,100.00%,100.00%


\begin{tabular}{lrrr}
\toprule
col & OVERALL & authors & title \\
model &  &  &  \\
\midrule
EmbDI_300 & 1.00 & 1.00 & 1.00 \\
EmbDI_512 & 1.00 & 1.00 & 1.00 \\
HRR_300 & 1.00 & 1.00 & 1.00 \\
HRR_512 & 1.00 & 1.00 & 1.00 \\
\bottomrule
\end{tabular}



col,OVERALL,authors,title,venue
model,,,,
EmbDI_300,100.00%,100.00%,100.00%,100.00%
EmbDI_512,99.33%,100.00%,100.00%,98.00%
HRR_300,100.00%,100.00%,100.00%,100.00%
HRR_512,100.00%,100.00%,100.00%,100.00%


\begin{tabular}{lrrrr}
\toprule
col & OVERALL & authors & title & venue \\
model &  &  &  &  \\
\midrule
EmbDI_300 & 1.00 & 1.00 & 1.00 & 1.00 \\
EmbDI_512 & 0.99 & 1.00 & 1.00 & 0.98 \\
HRR_300 & 1.00 & 1.00 & 1.00 & 1.00 \\
HRR_512 & 1.00 & 1.00 & 1.00 & 1.00 \\
\bottomrule
\end{tabular}



col,OVERALL,authors,title,venue,year
model,,,,,
EmbDI_300,82.50%,100.00%,98.00%,70.00%,62.00%
EmbDI_512,79.50%,100.00%,98.00%,64.00%,56.00%
HRR_300,100.00%,100.00%,100.00%,100.00%,100.00%
HRR_512,100.00%,100.00%,100.00%,100.00%,100.00%


\begin{tabular}{lrrrrr}
\toprule
col & OVERALL & authors & title & venue & year \\
model &  &  &  &  &  \\
\midrule
EmbDI_300 & 0.82 & 1.00 & 0.98 & 0.70 & 0.62 \\
EmbDI_512 & 0.80 & 1.00 & 0.98 & 0.64 & 0.56 \\
HRR_300 & 1.00 & 1.00 & 1.00 & 1.00 & 1.00 \\
HRR_512 & 1.00 & 1.00 & 1.00 & 1.00 & 1.00 \\
\bottomrule
\end{tabular}

